# Split Data (DT+/DT-) by distance from reference

Questo notebook esegue:
1. lettura di DN, DT-, DT+;
2. selezione di DT- e DT+ con distanza dalla reference in [25, 30];
3. split train/test al 15% per ciascun dataset DT- e DT+;
4. salvataggio dei FASTA richiesti;
5. filtro dei test split in [25, 30] e salvataggio in due file aggiuntivi;
6. split dei soli record con "vae" nell'header (DT- e DT+) in dentro/fuori [25,30] in una cartella separata.

## 1) Setup funzioni di utilita'

In questa sezione definiamo le funzioni riusabili per:
- calcolare la distanza di Hamming dalla sequenza di riferimento;
- salvare sequenze in formato FASTA;
- eseguire lo split train/test con random seed fisso.

In [ ]:
from pathlib import Path
import numpy as np

from adabmDCA.fasta import get_tokens, import_from_fasta


def hamming_distance_to_reference(seqs: np.ndarray, reference: np.ndarray) -> np.ndarray:
    if seqs.ndim != 2 or reference.ndim != 1:
        raise ValueError("`seqs` deve essere 2D e `reference` 1D")
    if seqs.shape[1] != reference.shape[0]:
        raise ValueError("Lunghezza sequenze e reference non compatibile")
    return (seqs != reference).sum(axis=1).astype(np.int32)


def write_fasta(headers: np.ndarray, seqs: np.ndarray, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)

    # If sequences are integer-encoded, decode using current RNA tokens.
    tok = globals().get("tokens", None)

    with out_path.open("w", encoding="utf-8") as f:
        for h, s in zip(headers, seqs):
            h_str = str(h)
            if not h_str.startswith(">"):
                h_str = ">" + h_str

            s_arr = np.asarray(s)
            if np.issubdtype(s_arr.dtype, np.integer):
                if tok is None:
                    raise ValueError("Token RNA non disponibili per decodificare sequenze intere")
                seq_str = "".join(tok[int(x)] for x in s_arr.tolist())
            else:
                seq_str = "".join(s_arr.astype(str).tolist())

            f.write(h_str + "\n")
            f.write(seq_str + "\n")


def split_train_test(headers: np.ndarray, seqs: np.ndarray, test_frac: float = 0.15, seed: int = 42):
    if not (0.0 < test_frac < 1.0):
        raise ValueError("`test_frac` deve essere in (0, 1)")

    n = seqs.shape[0]
    rng = np.random.default_rng(seed)
    idx = rng.permutation(n)

    n_test = int(np.floor(test_frac * n))
    n_test = max(1, n_test)

    test_idx = idx[:n_test]
    train_idx = idx[n_test:]

    return (
        headers[train_idx], seqs[train_idx],
        headers[test_idx], seqs[test_idx],
    )

## 2) Caricamento dataset e filtro per distanza [25, 30]

Qui leggiamo DN, DT- e DT+ dai file FASTA, scegliamo la reference come prima sequenza di DN, calcoliamo le distanze e teniamo i record DT-/DT+ con distanza compresa tra 25 e 30.

In [ ]:
# Lettura dati
tokens = get_tokens("rna")

headers_DN, data_DN = import_from_fasta(
    "../data/RF00028_aligned_no_inserts_reweighted_d10.fa",
    tokens=tokens,
    filter_sequences=True,
)
headers_DTm, data_DTm = import_from_fasta(
    "../data/DT-_197.fasta",
    tokens=tokens,
    filter_sequences=True,
)
headers_DTp, data_DTp = import_from_fasta(
   "../data/DT+_197.fasta",
    tokens=tokens,
    filter_sequences=True,
)

print("Shape DN :", data_DN.shape)
print("Shape DT-:", data_DTm.shape)
print("Shape DT+:", data_DTp.shape)

reference_seq = data_DN[0]

dist_tm = hamming_distance_to_reference(data_DTm, reference_seq)
dist_tp = hamming_distance_to_reference(data_DTp, reference_seq)

# Intervallo richiesto [25, 30]
low, high = 25, 30
mask_tm_25_30 = (dist_tm >= low) & (dist_tm <= high)
mask_tp_25_30 = (dist_tp >= low) & (dist_tp <= high)

headers_tm_25_30 = headers_DTm[mask_tm_25_30]
data_tm_25_30 = data_DTm[mask_tm_25_30]
headers_tp_25_30 = headers_DTp[mask_tp_25_30]
data_tp_25_30 = data_DTp[mask_tp_25_30]

print(f"DT- in [25,30]: {data_tm_25_30.shape[0]}")
print(f"DT+ in [25,30]: {data_tp_25_30.shape[0]}")

## 3) Inizializzazione cartella output

Creiamo la directory di output principale dove verranno scritti i file FASTA generati nelle sezioni successive.

In [ ]:
# # Salvataggio file richiesti
out_dir = Path("../data/split_data/TEST")
out_dir.mkdir(parents=True, exist_ok=True)

## 4) Blocco storico (commentato)

La cella seguente contiene codice commentato usato in precedenza per:
- split train/test classico su DT-/DT+;
- suddivisione dentro/fuori intervallo [25,30];
- salvataggio dei relativi file.

E' lasciato come riferimento, ma non viene eseguito.

In [ ]:



# def split_inside_outside(headers: np.ndarray, seqs: np.ndarray, reference: np.ndarray, low: int, high: int):
#     dist = hamming_distance_to_reference(seqs, reference)
#     mask_in = (dist >= low) & (dist <= high)
#     mask_out = ~mask_in
#     return (
#         headers[mask_in], seqs[mask_in],
#         headers[mask_out], seqs[mask_out],
#     )


# # -----------------------------------------------------------------------------
# # (A) Dataset globale: DT- e DT+ dentro/fuori [25,30]
# # -----------------------------------------------------------------------------
# path_tm_in_global = out_dir / "DTm_dist_25_30.fasta"
# path_tm_out_global = out_dir / "DTm_dist_outside_25_30.fasta"
# path_tp_in_global = out_dir / "DTp_dist_25_30.fasta"
# path_tp_out_global = out_dir / "DTp_dist_outside_25_30.fasta"

# write_fasta(headers_tm_25_30, data_tm_25_30, path_tm_in_global)
# write_fasta(headers_tp_25_30, data_tp_25_30, path_tp_in_global)

# mask_tm_out_25_30 = ~mask_tm_25_30
# mask_tp_out_25_30 = ~mask_tp_25_30
# headers_tm_out_25_30 = headers_DTm[mask_tm_out_25_30]
# data_tm_out_25_30 = data_DTm[mask_tm_out_25_30]
# headers_tp_out_25_30 = headers_DTp[mask_tp_out_25_30]
# data_tp_out_25_30 = data_DTp[mask_tp_out_25_30]

# write_fasta(headers_tm_out_25_30, data_tm_out_25_30, path_tm_out_global)
# write_fasta(headers_tp_out_25_30, data_tp_out_25_30, path_tp_out_global)

# # -----------------------------------------------------------------------------
# # (B) Split train/test al 15% separato per DT- e DT+
# # -----------------------------------------------------------------------------
# seed = 42
# tm_h_tr, tm_x_tr, tm_h_te, tm_x_te = split_train_test(headers_DTm, data_DTm, test_frac=0.15, seed=seed)
# tp_h_tr, tp_x_tr, tp_h_te, tp_x_te = split_train_test(headers_DTp, data_DTp, test_frac=0.15, seed=seed)

# # Salvo train/test separati per classe
# path_tm_train = out_dir / "DTm_train_split_85.fasta"
# path_tm_test = out_dir / "DTm_test_split_15.fasta"
# path_tp_train = out_dir / "DTp_train_split_85.fasta"
# path_tp_test = out_dir / "DTp_test_split_15.fasta"

# write_fasta(tm_h_tr, tm_x_tr, path_tm_train)
# write_fasta(tm_h_te, tm_x_te, path_tm_test)
# write_fasta(tp_h_tr, tp_x_tr, path_tp_train)
# write_fasta(tp_h_te, tp_x_te, path_tp_test)

# # -----------------------------------------------------------------------------
# # (C) Train/Test separati per classe e per intervallo dentro/fuori [25,30]
# # -----------------------------------------------------------------------------
# # DT- train
# (tm_h_tr_in, tm_x_tr_in, tm_h_tr_out, tm_x_tr_out) = split_inside_outside(
#     tm_h_tr, tm_x_tr, reference_seq, low, high
# )
# # DT- test
# (tm_h_te_in, tm_x_te_in, tm_h_te_out, tm_x_te_out) = split_inside_outside(
#     tm_h_te, tm_x_te, reference_seq, low, high
# )
# # DT+ train
# (tp_h_tr_in, tp_x_tr_in, tp_h_tr_out, tp_x_tr_out) = split_inside_outside(
#     tp_h_tr, tp_x_tr, reference_seq, low, high
# )
# # DT+ test
# (tp_h_te_in, tp_x_te_in, tp_h_te_out, tp_x_te_out) = split_inside_outside(
#     tp_h_te, tp_x_te, reference_seq, low, high
# )

# # Percorsi DT-
# path_tm_train_in = out_dir / "DTm_train_split_85_dist_25_30.fasta"
# path_tm_train_out = out_dir / "DTm_train_split_85_dist_outside_25_30.fasta"
# path_tm_test_in = out_dir / "DTm_test_split_15_dist_25_30.fasta"
# path_tm_test_out = out_dir / "DTm_test_split_15_dist_outside_25_30.fasta"

# # Percorsi DT+
# path_tp_train_in = out_dir / "DTp_train_split_85_dist_25_30.fasta"
# path_tp_train_out = out_dir / "DTp_train_split_85_dist_outside_25_30.fasta"
# path_tp_test_in = out_dir / "DTp_test_split_15_dist_25_30.fasta"
# path_tp_test_out = out_dir / "DTp_test_split_15_dist_outside_25_30.fasta"

# # Salvataggio DT-
# write_fasta(tm_h_tr_in, tm_x_tr_in, path_tm_train_in)
# write_fasta(tm_h_tr_out, tm_x_tr_out, path_tm_train_out)
# write_fasta(tm_h_te_in, tm_x_te_in, path_tm_test_in)
# write_fasta(tm_h_te_out, tm_x_te_out, path_tm_test_out)

# # Salvataggio DT+
# write_fasta(tp_h_tr_in, tp_x_tr_in, path_tp_train_in)
# write_fasta(tp_h_tr_out, tp_x_tr_out, path_tp_train_out)
# write_fasta(tp_h_te_in, tp_x_te_in, path_tp_test_in)
# write_fasta(tp_h_te_out, tp_x_te_out, path_tp_test_out)

# print("\nFile creati (globali dentro/fuori):")
# print(path_tm_in_global)
# print(path_tm_out_global)
# print(path_tp_in_global)
# print(path_tp_out_global)

# print("\nFile creati (split train/test per classe):")
# print(path_tm_train)
# print(path_tm_test)
# print(path_tp_train)
# print(path_tp_test)

# print("\nFile creati (split per classe dentro/fuori intervallo):")
# print(path_tm_train_in)
# print(path_tm_train_out)
# print(path_tm_test_in)
# print(path_tm_test_out)
# print(path_tp_train_in)
# print(path_tp_train_out)
# print(path_tp_test_in)
# print(path_tp_test_out)

# print("\nCardinalita globali:")
# print(f"DT- in [25,30]: {data_tm_25_30.shape[0]} | fuori: {data_tm_out_25_30.shape[0]}")
# print(f"DT+ in [25,30]: {data_tp_25_30.shape[0]} | fuori: {data_tp_out_25_30.shape[0]}")

# print("\nCardinalita split per classe:")
# print(f"DT- train/test: {tm_x_tr.shape[0]} / {tm_x_te.shape[0]}")
# print(f"DT+ train/test: {tp_x_tr.shape[0]} / {tp_x_te.shape[0]}")

# print("\nCardinalita split DT- (inside/outside):")
# print(f"train: {tm_x_tr_in.shape[0]} / {tm_x_tr_out.shape[0]}")
# print(f"test : {tm_x_te_in.shape[0]} / {tm_x_te_out.shape[0]}")

# print("\nCardinalita split DT+ (inside/outside):")
# print(f"train: {tp_x_tr_in.shape[0]} / {tp_x_tr_out.shape[0]}")
# print(f"test : {tp_x_te_in.shape[0]} / {tp_x_te_out.shape[0]}")

## 5) Costruzione split VAE-plus

Da qui inizia il flusso principale attivo per creare il dataset `vae_plus_split`:
- identificazione sequenze con `vae` nell'header;
- separazione tra VAE e non-VAE;
- preparazione cartella dedicata.

In [ ]:
# -----------------------------------------------------------------------------
# (D) Split VAE in cartella separata: distanza dalla reference dentro/fuori [25,30]
# -----------------------------------------------------------------------------
vae_dir = out_dir / "vae_plus_split"
vae_dir.mkdir(parents=True, exist_ok=True)


def header_contains_vae(headers: np.ndarray) -> np.ndarray:
    return np.array(["vae" in str(h).lower() for h in headers], dtype=bool)

# mask for VAE seqs
mask_tm_vae = header_contains_vae(headers_DTm)
mask_tp_vae = header_contains_vae(headers_DTp)
# headers and data of VAE
headers_tm_vae = headers_DTm[mask_tm_vae]
data_tm_vae = data_DTm[mask_tm_vae]
headers_tp_vae = headers_DTp[mask_tp_vae]
data_tp_vae = data_DTp[mask_tp_vae]
# headers and data of NOT VAE
headers_tm_other = headers_DTm[~mask_tm_vae]
data_tm_other = data_DTm[~mask_tm_vae]
headers_tp_other = headers_DTp[~mask_tp_vae]
data_tp_other = data_DTp[~mask_tp_vae]

### 5.1) Matrici di distanza VAE vs non-VAE

Nella prossima cella calcoliamo le distanze di Hamming tra ogni sequenza VAE e tutte le sequenze non-VAE (sia intra-classe che cross-classe).

In [ ]:
# compute distance between TM VAE and others
h_tm_vae_tm = np.zeros((data_tm_vae.shape[0], data_tm_other.shape[0]))
h_tm_vae_tp = np.zeros((data_tm_vae.shape[0], data_tp_other.shape[0]))
for i,seq in enumerate(data_tm_vae):
    data_tm_vae_broad_tm = np.broadcast_to(seq, data_tm_other.shape)
    data_tm_vae_broad_tp = np.broadcast_to(seq, data_tp_other.shape)
    h_tm_vae_tm[i, :] = np.sum((data_tm_vae_broad_tm != data_tm_other), axis=1)
    h_tm_vae_tp[i, :] = np.sum((data_tm_vae_broad_tp != data_tp_other), axis=1)

# compute distance between TP VAE and others
h_tp_vae_tm = np.zeros((data_tp_vae.shape[0], data_tm_other.shape[0]))
h_tp_vae_tp = np.zeros((data_tp_vae.shape[0], data_tp_other.shape[0]))
for i,seq in enumerate(data_tp_vae):
    data_tp_vae_broad_tm = np.broadcast_to(seq, data_tm_other.shape)
    data_tp_vae_broad_tp = np.broadcast_to(seq, data_tp_other.shape)

    h_tp_vae_tm[i, :] = np.sum((data_tp_vae_broad_tm != data_tm_other), axis=1)
    h_tp_vae_tp[i, :] = np.sum((data_tp_vae_broad_tp != data_tp_other), axis=1)

### 5.2) Rimozione sequenze troppo vicine ai VAE

La prossima cella costruisce le maschere di esclusione: una sequenza non-VAE viene marcata se ha distanza <= 10 da almeno una sequenza VAE (DT- o DT+).

In [ ]:
# mask sequences in other colser than 10 hamming from every seq in VAE
mask_remove_from_tp_other = ((h_tp_vae_tp <= 10).sum(axis=0) != 0) | ((h_tm_vae_tp <= 10).sum(axis=0) != 0)
mask_remove_from_tm_other = ((h_tp_vae_tm <= 10).sum(axis=0) != 0) | ((h_tm_vae_tm <= 10).sum(axis=0) != 0)

# headers and data of removed sequences
header_remove_tp_other = headers_tp_other[mask_remove_from_tp_other]
data_remove_tp_other = data_tp_other[mask_remove_from_tp_other]

header_remove_tm_other = headers_tm_other[mask_remove_from_tm_other]
data_remove_tm_other = data_tm_other[mask_remove_from_tm_other]

### 5.3) Costruzione validation VAE-plus e train residuo

Nella cella seguente uniamo VAE + sequenze rimosse per creare il validation set (`VAE_plus`) e manteniamo il complemento come training non-VAE pulito.

In [ ]:
# create validation set by: unifying VAE and removed, and removing from OTHER
header_val_tm_vae_plus = np.concatenate((headers_tm_vae, header_remove_tm_other))
data_val_tm_vae_plus = np.concatenate((data_tm_vae, data_remove_tm_other))

header_val_tp_vae_plus = np.concatenate((headers_tp_vae, header_remove_tp_other))
data_val_tp_vae_plus = np.concatenate((data_tp_vae, data_remove_tp_other))


header_tm_other_minus = headers_tm_other[~mask_remove_from_tm_other]
data_tm_other_minus = data_tm_other[~mask_remove_from_tm_other]

header_tp_other_minus = headers_tp_other[~mask_remove_from_tp_other]
data_tp_other_minus = data_tp_other[~mask_remove_from_tp_other]


### 5.4) Filtro del validation su distanza [25, 30]

La prossima cella calcola la distanza dalla reference per il validation `VAE_plus` e crea anche la versione ristretta all'intervallo [25,30].

In [ ]:
# hamming distance from reference

dist_val_vae_tp = hamming_distance_to_reference(data_val_tp_vae_plus, reference_seq)
dist_val_vae_tm = hamming_distance_to_reference(data_val_tm_vae_plus, reference_seq)


# dist_val_others_tp = hamming_distance_to_reference(data_val_tp_other_minus, reference_seq)
# dist_val_others_tm = hamming_distance_to_reference(data_val_tm_other_minus, reference_seq)

mask_d25_30_val_vae_tp = (25 <= dist_val_vae_tp) & (dist_val_vae_tp <= 30)
mask_d25_30_val_vae_tm = (25 <= dist_val_vae_tm) & (dist_val_vae_tm <= 30)


header_val_tm_vae_plus_d25_30 = header_val_tm_vae_plus[mask_d25_30_val_vae_tm]
data_val_tm_vae_plus_d25_30 = data_val_tm_vae_plus[mask_d25_30_val_vae_tm]

header_val_tp_vae_plus_d25_30 = header_val_tp_vae_plus[mask_d25_30_val_vae_tp]
data_val_tp_vae_plus_d25_30 = data_val_tp_vae_plus[mask_d25_30_val_vae_tp]

### 5.5) Salvataggio FASTA finali

Nella prossima cella vengono scritti i file FASTA finali per:
- validation `VAE_plus` (completo);
- training non-VAE pulito;
- versioni filtrate in [25,30].

In [ ]:
path_val_tm_vae_plus = vae_dir / "DTm_val_VAE_plus.fasta"
path_tm_vae_out_minus = vae_dir / "DTm_train_non_vae_minus.fasta"

path_val_tp_vae_plus = vae_dir / "DTp_val_VAE_plus.fasta"
path_tp_vae_out_minus = vae_dir / "DTp_train_non_vae_minus.fasta"

path_val_tm_vae_25_30 = vae_dir / "DTm_val_VAE_plus_25_30.fasta"
path_val_tp_vae_25_30 = vae_dir / "DTp_val_VAE_plus_25_30.fasta"

write_fasta(header_val_tm_vae_plus, data_val_tm_vae_plus, path_val_tm_vae_plus)
write_fasta(header_tm_other_minus, data_tm_other_minus, path_tm_vae_out_minus)

write_fasta(header_val_tp_vae_plus, data_val_tp_vae_plus, path_val_tp_vae_plus)
write_fasta(header_tp_other_minus, data_tp_other_minus, path_tp_vae_out_minus)

write_fasta(header_val_tm_vae_plus_d25_30, data_val_tm_vae_plus_d25_30, path_val_tm_vae_25_30)
write_fasta(header_val_tp_vae_plus_d25_30, data_val_tp_vae_plus_d25_30, path_val_tp_vae_25_30)

## 7) Blocco legacy VAE inside/outside

La cella seguente e' uno snippet legacy per lo split VAE dentro/fuori [25,30].
Nota: dipende da `split_inside_outside`, che nel notebook attuale e' commentata nel blocco storico.

In [ ]:


# (tm_h_vae_in, tm_x_vae_in, tm_h_vae_out, tm_x_vae_out) = split_inside_outside(
#     headers_tm_vae, data_tm_vae, reference_seq, low, high
# )
# (tp_h_vae_in, tp_x_vae_in, tp_h_vae_out, tp_x_vae_out) = split_inside_outside(
#     headers_tp_vae, data_tp_vae, reference_seq, low, high
# )

# path_tm_vae_in = vae_dir / "DTm_vae_dist_25_30.fasta"
# path_tm_vae_out = vae_dir / "DTm_vae_dist_outside_25_30.fasta"
# path_tp_vae_in = vae_dir / "DTp_vae_dist_25_30.fasta"
# path_tp_vae_out = vae_dir / "DTp_vae_dist_outside_25_30.fasta"

# write_fasta(tm_h_vae_in, tm_x_vae_in, path_tm_vae_in)
# write_fasta(tm_h_vae_out, tm_x_vae_out, path_tm_vae_out)
# write_fasta(tp_h_vae_in, tp_x_vae_in, path_tp_vae_in)
# write_fasta(tp_h_vae_out, tp_x_vae_out, path_tp_vae_out)

# print("\nSplit VAE creato in:", vae_dir)
# print("File VAE DT-:", path_tm_vae_in, "|", path_tm_vae_out)
# print("File VAE DT+:", path_tp_vae_in, "|", path_tp_vae_out)
# print("Cardinalita VAE DT- in/out:", tm_x_vae_in.shape[0], "/", tm_x_vae_out.shape[0])
# print("Cardinalita VAE DT+ in/out:", tp_x_vae_in.shape[0], "/", tp_x_vae_out.shape[0])

## 8) Split 0.15 con VAE [25,30] forzate nel test

Workflow richiesto:
1. Prendiamo DT- e DT+.
2. Escludiamo temporaneamente le sequenze con header `vae` e distanza dalla reference in `[25,30]`.
3. Eseguiamo split train/test classico (`test_frac=0.15`) sui rimanenti, separatamente per DT- e DT+.
4. Aggiungiamo al test le sequenze escluse al punto 2.
5. Salviamo tutto in `../data/split_data_197/split_0.15_vae25_30_in_test`.

In [ ]:
# Configurazione output
out_dir_vae_test = Path("../data/split_train_validation/split_0.15_vae25_30_in_test")
out_dir_vae_test.mkdir(parents=True, exist_ok=True)

# Parametri split
seed = 42
test_frac = 0.15
low, high = 25, 30

# Maschera VAE su header
mask_tm_vae_all = np.array(["vae" in str(h).lower() for h in headers_DTm], dtype=bool)
mask_tp_vae_all = np.array(["vae" in str(h).lower() for h in headers_DTp], dtype=bool)

# Maschera distanza [25,30] dalla reference (gia' calcolata sopra su DT-/DT+ completi)
mask_tm_d25_30_all = (dist_tm >= low) & (dist_tm <= high)
mask_tp_d25_30_all = (dist_tp >= low) & (dist_tp <= high)

# Sequenze da escludere inizialmente e poi aggiungere nel test
mask_tm_excluded = mask_tm_vae_all & mask_tm_d25_30_all
mask_tp_excluded = mask_tp_vae_all & mask_tp_d25_30_all

headers_tm_excluded = headers_DTm[mask_tm_excluded]
data_tm_excluded = data_DTm[mask_tm_excluded]
headers_tp_excluded = headers_DTp[mask_tp_excluded]
data_tp_excluded = data_DTp[mask_tp_excluded]

# Dataset rimanenti su cui fare lo split classico
headers_tm_remaining = headers_DTm[~mask_tm_excluded]
data_tm_remaining = data_DTm[~mask_tm_excluded]
headers_tp_remaining = headers_DTp[~mask_tp_excluded]
data_tp_remaining = data_DTp[~mask_tp_excluded]

if data_tm_remaining.shape[0] == 0 or data_tp_remaining.shape[0] == 0:
    raise ValueError("Dopo l'esclusione VAE [25,30], DT- o DT+ e' vuoto: impossibile fare split.")

# Split classico 85/15 sui rimanenti, separato per classe
(tm_h_tr, tm_x_tr, tm_h_te, tm_x_te) = split_train_test(
    headers_tm_remaining,
    data_tm_remaining,
    test_frac=test_frac,
    seed=seed,
)
(tp_h_tr, tp_x_tr, tp_h_te, tp_x_te) = split_train_test(
    headers_tp_remaining,
    data_tp_remaining,
    test_frac=test_frac,
    seed=seed,
)

# Aggiunta delle sequenze escluse nel test finale
tm_h_te_final = np.concatenate((tm_h_te, headers_tm_excluded), axis=0)
tm_x_te_final = np.concatenate((tm_x_te, data_tm_excluded), axis=0)
tp_h_te_final = np.concatenate((tp_h_te, headers_tp_excluded), axis=0)
tp_x_te_final = np.concatenate((tp_x_te, data_tp_excluded), axis=0)

# Percorsi output
path_tm_train = out_dir_vae_test / "DTm_train_split_85_excluding_vae_25_30.fasta"
path_tm_test = out_dir_vae_test / "DTm_test_split_15_plus_excluded_vae_25_30.fasta"
path_tp_train = out_dir_vae_test / "DTp_train_split_85_excluding_vae_25_30.fasta"
path_tp_test = out_dir_vae_test / "DTp_test_split_15_plus_excluded_vae_25_30.fasta"

# Salvataggio split finali
write_fasta(tm_h_tr, tm_x_tr, path_tm_train)
write_fasta(tm_h_te_final, tm_x_te_final, path_tm_test)
write_fasta(tp_h_tr, tp_x_tr, path_tp_train)
write_fasta(tp_h_te_final, tp_x_te_final, path_tp_test)

# Salvataggio opzionale dei soli esclusi (tracciabilita')
path_tm_excluded = out_dir_vae_test / "DTm_excluded_vae_25_30_added_to_test.fasta"
path_tp_excluded = out_dir_vae_test / "DTp_excluded_vae_25_30_added_to_test.fasta"
write_fasta(headers_tm_excluded, data_tm_excluded, path_tm_excluded)
write_fasta(headers_tp_excluded, data_tp_excluded, path_tp_excluded)

print("Output directory:", out_dir_vae_test)
print("\nDT-")
print(f"Totale iniziale: {data_DTm.shape[0]}")
print(f"Escluse VAE in [25,30]: {data_tm_excluded.shape[0]}")
print(f"Rimanenti per split: {data_tm_remaining.shape[0]}")
print(f"Train finale: {tm_x_tr.shape[0]}")
print(f"Test finale (split + escluse): {tm_x_te_final.shape[0]}")

print("\nDT+")
print(f"Totale iniziale: {data_DTp.shape[0]}")
print(f"Escluse VAE in [25,30]: {data_tp_excluded.shape[0]}")
print(f"Rimanenti per split: {data_tp_remaining.shape[0]}")
print(f"Train finale: {tp_x_tr.shape[0]}")
print(f"Test finale (split + escluse): {tp_x_te_final.shape[0]}")

print("\nFile creati:")
print(path_tm_train)
print(path_tm_test)
print(path_tp_train)
print(path_tp_test)
print(path_tm_excluded)
print(path_tp_excluded)

## 9) Istogrammi distanze da reference (train vs test)

In questa sezione plottiamo la distribuzione delle distanze di Hamming dalla reference per train e test finali.
I bin sono interi: ogni valore di distanza ha il suo bin dedicato.

In [ ]:
import matplotlib.pyplot as plt

# Distanze dalla reference per split finale (sezione 8)
dist_tm_train = hamming_distance_to_reference(tm_x_tr, reference_seq)
dist_tm_test = hamming_distance_to_reference(tm_x_te_final, reference_seq)
dist_tp_train = hamming_distance_to_reference(tp_x_tr, reference_seq)
dist_tp_test = hamming_distance_to_reference(tp_x_te_final, reference_seq)

# Bin interi: 1 bin per ogni possibile valore di distanza
all_dist = np.concatenate((dist_tm_train, dist_tm_test, dist_tp_train, dist_tp_test))
min_d = int(all_dist.min())
max_d = int(all_dist.max())
bins = np.arange(min_d - 0.5, max_d + 1.5, 1)

fig, axes = plt.subplots(1, 2, figsize=(28, 5), sharey=True)

axes[0].hist(dist_tm_train, bins=bins, alpha=0.6, label="DT- train", color="tab:blue")
axes[0].hist(dist_tm_test, bins=bins, alpha=0.6, label="DT- test", color="tab:orange")
axes[0].set_title("DT-: distanza da reference")
axes[0].set_xlabel("Distanza di Hamming")
axes[0].set_ylabel("Conteggio")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.25)

axes[1].hist(dist_tp_train, bins=bins, alpha=0.6, label="DT+ train", color="tab:green")
axes[1].hist(dist_tp_test, bins=bins, alpha=0.6, label="DT+ test", color="tab:red")
axes[1].set_title("DT+: distanza da reference")
axes[1].set_xlabel("Distanza di Hamming")
axes[1].legend()
axes[1].grid(axis="y", alpha=0.25)

for ax in axes:
    ax.set_xticks(np.arange(min_d, max_d + 1, 1))

plt.tight_layout()
plt.show()

## 10) Remove unlikely sequences for the model

In questa sezione:
1. carichiamo lo stesso modello usato in `src/dpo_train.py`;
2. calcoliamo la NLL per tutte le sequenze DT- e DT+;
3. visualizziamo NLL vs distanza dalla reference con densita` dei punti;
4. impostiamo una soglia NLL manuale (o percentile);
5. filtriamo le DT- con NLL sopra soglia;
6. rifacciamo lo split 0.15 con validation forzata contenente VAE in [25,30] **piu`** tutte le sequenze entro 10 mismatch da queste.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.transformer import GPTTransformer
from src.dpo_config import CONFIG
from src.dpo_data import _build_labels_from_inputs


def decode_int_encoded_sequences(seqs: np.ndarray, token_list) -> list[str]:
    decoded = []
    for s in seqs:
        arr = np.asarray(s)
        if np.issubdtype(arr.dtype, np.integer):
            decoded.append("".join(token_list[int(x)] for x in arr.tolist()))
        else:
            decoded.append("".join(arr.astype(str).tolist()))
    return decoded


def build_stoi_from_dn_fasta(dn_fasta_path: Path):
    with dn_fasta_path.open("r", encoding="utf-8") as f:
        seq_lines = f.read().splitlines()[1::2]
    all_chars = "\n".join(seq_lines)
    chars = sorted(list(set(all_chars)))
    pad_symbol = "?"
    chars = chars + [pad_symbol]
    stoi = {ch: i for i, ch in enumerate(chars)}
    return stoi, stoi[pad_symbol]


def encode_with_boundaries(seq: str, stoi: dict[str, int], block_size: int, pad_token: int) -> torch.Tensor:
    txt = f"\n{seq}\n"
    ids = [stoi[c] for c in txt]
    if len(ids) < block_size:
        ids += [pad_token] * (block_size - len(ids))
    else:
        ids = ids[:block_size]
    return torch.tensor(ids, dtype=torch.long)


def compute_sequence_token_nll(model, encoded_data: list[torch.Tensor], pad_token: int, device: torch.device, batch_size: int = 256) -> np.ndarray:
    nll_all = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(encoded_data), batch_size):
            chunk = encoded_data[start : start + batch_size]
            x = torch.stack(chunk, dim=0).to(device)
            y = _build_labels_from_inputs(x, pad_token)

            logits = model(x)["logits"]
            log_probs = F.log_softmax(logits, dim=-1)
            per_token_logp = torch.gather(log_probs, 2, y.unsqueeze(2)).squeeze(2)
            mask = (y != pad_token)

            nll_sum = -(per_token_logp * mask).sum(dim=1)
            token_count = mask.sum(dim=1).clamp_min(1)
            nll_token_mean = (nll_sum / token_count).detach().cpu().numpy()
            nll_all.append(nll_token_mean)

    if len(nll_all) == 0:
        return np.array([], dtype=np.float32)
    return np.concatenate(nll_all, axis=0)


def min_hamming_to_pool(query: np.ndarray, pool: np.ndarray) -> np.ndarray:
    if pool.shape[0] == 0:
        return np.full(query.shape[0], np.inf)
    mins = np.empty(query.shape[0], dtype=np.float32)
    for i, s in enumerate(query):
        mins[i] = np.min(np.sum(pool != s, axis=1))
    return mins

In [ ]:
# Caricamento modello coerente con src/dpo_train.py
project_dir = Path("..").resolve()
dn_path_for_vocab = project_dir / CONFIG.dn_path
ckpt_path = project_dir / CONFIG.pretrained_ckpt_path

if not dn_path_for_vocab.exists():
    raise FileNotFoundError(f"DN path non trovato: {dn_path_for_vocab}")
if not ckpt_path.exists():
    raise FileNotFoundError(f"Checkpoint non trovato: {ckpt_path}")

stoi, pad_token_model = build_stoi_from_dn_fasta(dn_path_for_vocab)
vocab_size_model = len(stoi)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

model_nll = GPTTransformer(
    vocab_size=vocab_size_model,
    embed_dim=CONFIG.n_embd,
    num_layers=CONFIG.n_layer,
    num_heads=CONFIG.n_head,
    mlp_ratio=4,
    dropout_p=0.1,
    pad_id=pad_token_model,
)
state_dict = torch.load(ckpt_path, map_location="cpu")
model_nll.load_state_dict(state_dict)
model_nll = model_nll.to(device)
model_nll.eval()

print("Model loaded from:", ckpt_path)
print("Device:", device)
print("Vocab size:", vocab_size_model)
print("Pad token id:", pad_token_model)

In [ ]:
# NLL su tutte le sequenze DT- e DT+
if "tokens" not in globals():
    tokens = get_tokens("rna")

seqs_tm_str = decode_int_encoded_sequences(data_DTm, tokens)
seqs_tp_str = decode_int_encoded_sequences(data_DTp, tokens)

encoded_tm = [
    encode_with_boundaries(s, stoi=stoi, block_size=CONFIG.max_context, pad_token=pad_token_model)
    for s in seqs_tm_str
]
encoded_tp = [
    encode_with_boundaries(s, stoi=stoi, block_size=CONFIG.max_context, pad_token=pad_token_model)
    for s in seqs_tp_str
]

nll_tm = compute_sequence_token_nll(
    model_nll,
    encoded_tm,
    pad_token=pad_token_model,
    device=device,
    batch_size=256,
)
nll_tp = compute_sequence_token_nll(
    model_nll,
    encoded_tp,
    pad_token=pad_token_model,
    device=device,
    batch_size=256,
)

df_nll = pd.DataFrame({
    "class": np.concatenate([np.full(len(nll_tm), "DT-"), np.full(len(nll_tp), "DT+")]),
    "distance": np.concatenate([dist_tm, dist_tp]).astype(float),
    "nll": np.concatenate([nll_tm, nll_tp]).astype(float),
})

print("DT- NLL shape:", nll_tm.shape)
print("DT+ NLL shape:", nll_tp.shape)
print(df_nll.groupby("class")["nll"].describe())

In [ ]:
# Selezione soglia NLL (manuale o percentile)
USE_MANUAL_NLL_THRESHOLD = False
NLL_THRESHOLD_MANUAL = 1.60

# Esempio richiesto: prova con percentile=10; valori piu` alti (es. 80-90) sono meno aggressivi
NLL_THRESHOLD_PERCENTILE = 60

if USE_MANUAL_NLL_THRESHOLD:
    nll_threshold = float(NLL_THRESHOLD_MANUAL)
else:
    nll_threshold = float(np.percentile(nll_tm, NLL_THRESHOLD_PERCENTILE))

mask_tm_high_nll = nll_tm > nll_threshold

print(f"Soglia NLL selezionata: {nll_threshold:.6f}")
print(f"DT- totali: {len(nll_tm)}")
print(f"DT- con NLL > soglia (da filtrare): {int(mask_tm_high_nll.sum())}")
print(f"DT- mantenute dopo filtro NLL: {int((~mask_tm_high_nll).sum())}")

In [ ]:
# Plot distanza vs NLL: scatter + densita` + istogrammi DT-/DT+ su asse Y
fig = plt.figure(figsize=(20, 6), constrained_layout=True)
gs = fig.add_gridspec(1, 3, width_ratios=[1.2, 1.2, 0.8])

ax_scatter = fig.add_subplot(gs[0, 0])
ax_density = fig.add_subplot(gs[0, 1], sharey=ax_scatter)
ax_hist_y = fig.add_subplot(gs[0, 2], sharey=ax_scatter)

# Scatter con classi separate
ax_scatter.scatter(dist_tm, nll_tm, s=16, alpha=0.35, c="red", label="DT-")
ax_scatter.scatter(dist_tp, nll_tp, s=16, alpha=0.35, c="green", label="DT+")
ax_scatter.set_title("NLL vs distanza dalla reference")
ax_scatter.set_xlabel("Distanza di Hamming")
ax_scatter.set_ylabel("NLL media per token")
ax_scatter.legend()
ax_scatter.grid(alpha=0.2)

# Densita` con hexbin separati per classe
hb_tm = ax_density.hexbin(dist_tm, nll_tm, gridsize=28, mincnt=1, cmap="Reds", alpha=0.65)
hb_tp = ax_density.hexbin(dist_tp, nll_tp, gridsize=28, mincnt=1, cmap="Greens", alpha=0.65)
ax_density.set_title("Densita` punti (DT- rosso, DT+ verde)")
ax_density.set_xlabel("Distanza di Hamming")
ax_density.grid(alpha=0.2)

# Istogrammi DT- e DT+ con asse Y condiviso (NLL)
bins_y = np.linspace(min(nll_tm.min(), nll_tp.min()), max(nll_tm.max(), nll_tp.max()), 40)
nll_tp_max = float(nll_tp.max())
nll_tp_p99 = float(np.percentile(nll_tp, 99.9))
ax_hist_y.hist(nll_tm, bins=bins_y, orientation="horizontal", color="red", alpha=0.45, edgecolor="none", label="DT-")
ax_hist_y.hist(nll_tp, bins=bins_y, orientation="horizontal", color="green", alpha=0.35, edgecolor="none", label="DT+")
ax_hist_y.set_title("Istogrammi NLL DT-/DT+ (asse Y = NLL)")
ax_hist_y.set_xlabel("Conteggio")
ax_hist_y.legend()
ax_hist_y.grid(alpha=0.2, axis="x")

# Linea soglia su tutta la figura (stesso asse Y condiviso)
for ax in (ax_scatter, ax_density, ax_hist_y):
    ax.axhline(nll_threshold, color="black", linestyle="--", linewidth=2)
    ax.axhline(nll_tp_max, color="green", linestyle=":", linewidth=2)
    ax.axhline(nll_tp_p99, color="blue", linestyle="-.", linewidth=2)

# Etichetta soglia nell'ultimo pannello
ax_hist_y.text(
    0.98,
    nll_threshold,
    f"  soglia={nll_threshold:.3f}",
    va="bottom",
    ha="right",
    transform=ax_hist_y.get_yaxis_transform(),
    color="black",
    fontsize=10,
)
ax_hist_y.text(
    0.98,
    nll_tp_max,
    f"  max DT+={nll_tp_max:.3f}",
    va="bottom",
    ha="right",
    transform=ax_hist_y.get_yaxis_transform(),
    color="green",
    fontsize=10,
)
ax_hist_y.text(
    0.98,
    nll_tp_p99,
    f"  p99,9 DT+={nll_tp_p99:.3f}",
    va="bottom",
    ha="right",
    transform=ax_hist_y.get_yaxis_transform(),
    color="blue",
    fontsize=10,
)

cbar_tm = fig.colorbar(hb_tm, ax=ax_density, fraction=0.046, pad=0.04)
cbar_tm.set_label("Count DT-")
cbar_tp = fig.colorbar(hb_tp, ax=ax_density, fraction=0.046, pad=0.12)
cbar_tp.set_label("Count DT+")
plt.savefig( "nll_vs_distance_scatter_density_hist_soglia_60p_of_DT-.png", dpi=300)
plt.show()

In [ ]:
# Split 0.15 stile sezione 8 + VAE[25,30] + vicine (<=10) forzate in validation,
# con filtro NLL sulle sole DT-
out_dir_section10 = Path(f"../data/split_train_validation/split_fraction-0.15_ add-vae-25-30-and-close-to-validation_nll-filtered_{NLL_THRESHOLD_PERCENTILE}")
out_dir_section10.mkdir(parents=True, exist_ok=True)

seed = 42
test_frac = 0.15
low, high = 25, 30

# 1) VAE in [25,30] da forzare in validation
mask_tm_vae = np.array(["vae" in str(h).lower() for h in headers_DTm], dtype=bool)
mask_tp_vae = np.array(["vae" in str(h).lower() for h in headers_DTp], dtype=bool)
mask_tm_vae_25_30 = mask_tm_vae & (dist_tm >= low) & (dist_tm <= high)
mask_tp_vae_25_30 = mask_tp_vae & (dist_tp >= low) & (dist_tp <= high)

# 2) Pool VAE[25,30] (DT- + DT+) e sequenze vicine (<=10 mismatch)
vae_pool = np.concatenate(
    [
        data_DTm[mask_tm_vae_25_30],
        data_DTp[mask_tp_vae_25_30],
    ],
    axis=0,
)
min_dist_tm_to_vae = min_hamming_to_pool(data_DTm, vae_pool)
min_dist_tp_to_vae = min_hamming_to_pool(data_DTp, vae_pool)

mask_tm_close_to_vae = min_dist_tm_to_vae <= 10
mask_tp_close_to_vae = min_dist_tp_to_vae <= 10

# 3) Set da forzare in validation (VAE[25,30] + vicine)
mask_tm_force_val = mask_tm_vae_25_30 | mask_tm_close_to_vae
mask_tp_force_val = mask_tp_vae_25_30 | mask_tp_close_to_vae

# 4) Filtro NLL su DT- (rimozione definitiva)
mask_tm_drop_nll = mask_tm_high_nll

# 5) Costruzione rimanenti per split 85/15
mask_tm_remaining = (~mask_tm_force_val) & (~mask_tm_drop_nll)
mask_tp_remaining = (~mask_tp_force_val)

headers_tm_remaining = headers_DTm[mask_tm_remaining]
data_tm_remaining = data_DTm[mask_tm_remaining]
headers_tp_remaining = headers_DTp[mask_tp_remaining]
data_tp_remaining = data_DTp[mask_tp_remaining]

if data_tm_remaining.shape[0] == 0 or data_tp_remaining.shape[0] == 0:
    raise ValueError("Dataset rimanente vuoto dopo filtro NLL/maschere VAE-close: impossibile fare split.")

# 6) Split classico sui rimanenti
(tm_h_tr, tm_x_tr, tm_h_te, tm_x_te) = split_train_test(
    headers_tm_remaining,
    data_tm_remaining,
    test_frac=test_frac,
    seed=seed,
)
(tp_h_tr, tp_x_tr, tp_h_te, tp_x_te) = split_train_test(
    headers_tp_remaining,
    data_tp_remaining,
    test_frac=test_frac,
    seed=seed,
)

# 7) Validation forzata aggiunta al test split
headers_tm_force_val = headers_DTm[mask_tm_force_val & (~mask_tm_drop_nll)]
data_tm_force_val = data_DTm[mask_tm_force_val & (~mask_tm_drop_nll)]
headers_tp_force_val = headers_DTp[mask_tp_force_val]
data_tp_force_val = data_DTp[mask_tp_force_val]

tm_h_val_final = np.concatenate((tm_h_te, headers_tm_force_val), axis=0)
tm_x_val_final = np.concatenate((tm_x_te, data_tm_force_val), axis=0)
tp_h_val_final = np.concatenate((tp_h_te, headers_tp_force_val), axis=0)
tp_x_val_final = np.concatenate((tp_x_te, data_tp_force_val), axis=0)

# 8) Salvataggi principali
path_tm_train = out_dir_section10 / "DTm_train_after-nll-filter.fasta"
path_tm_val = out_dir_section10 / "DTm_val_plus-vae25-30-and-close.fasta"
path_tp_train = out_dir_section10 / "DTp_train.fasta"
path_tp_val = out_dir_section10 / "DTp_val_plus-vae25-30-and-close.fasta"

write_fasta(tm_h_tr, tm_x_tr, path_tm_train)
write_fasta(tm_h_val_final, tm_x_val_final, path_tm_val)
write_fasta(tp_h_tr, tp_x_tr, path_tp_train)
write_fasta(tp_h_val_final, tp_x_val_final, path_tp_val)

# 9) Salvataggi separati richiesti: VAE + vicine
path_tm_vae_plus_close = out_dir_section10 / "DTm_vae25-30_plus-close-close.fasta"
path_tp_vae_plus_close = out_dir_section10 / "DTp_vae25-30_plus-close-close.fasta"
write_fasta(headers_tm_force_val, data_tm_force_val, path_tm_vae_plus_close)
write_fasta(headers_tp_force_val, data_tp_force_val, path_tp_vae_plus_close)

# 10) Salvataggio DT- rimosse per NLL alta (tracciabilita`)
path_tm_removed_nll = out_dir_section10 / "DTm_removed-high-nll.fasta"
write_fasta(headers_DTm[mask_tm_drop_nll], data_DTm[mask_tm_drop_nll], path_tm_removed_nll)

print("Output directory:", out_dir_section10)
print("\nSoglia NLL DT-:", nll_threshold)
print("DT- rimosse per NLL alta:", int(mask_tm_drop_nll.sum()))

print("\nDT- forzate in validation (VAE[25,30] + vicine <=10, dopo filtro NLL):", headers_tm_force_val.shape[0])
print("DT+ forzate in validation (VAE[25,30] + vicine <=10):", headers_tp_force_val.shape[0])

print("\nCardinalita` finali")
print(f"DT- train / val: {tm_x_tr.shape[0]} / {tm_x_val_final.shape[0]}")
print(f"DT+ train / val: {tp_x_tr.shape[0]} / {tp_x_val_final.shape[0]}")

print("\nFile creati:")
print(path_tm_train)
print(path_tm_val)
print(path_tp_train)
print(path_tp_val)
print(path_tm_vae_plus_close)
print(path_tp_vae_plus_close)
print(path_tm_removed_nll)

In [ ]:
# Plot di conferma: distanza minima dalle VAE [25,30] per sequenze escluse vs altre
# Definiamo "escluse" come le sequenze non-VAE che sono vicine (<=10) alle VAE [25,30].
if vae_pool.shape[0] == 0:
    raise ValueError("Nessuna sequenza VAE in [25,30]: impossibile costruire il plot di conferma.")

# Distanze minime gia` calcolate in sezione 10:
# - min_dist_tm_to_vae: per tutte le DT-
# - min_dist_tp_to_vae: per tutte le DT+

# Evitiamo che le VAE stesse (distanza 0) dominino il confronto
mask_tm_vae_25_30_only = mask_tm_vae_25_30
mask_tp_vae_25_30_only = mask_tp_vae_25_30

mask_tm_excluded_non_vae = mask_tm_close_to_vae & (~mask_tm_vae_25_30_only)
mask_tp_excluded_non_vae = mask_tp_close_to_vae & (~mask_tp_vae_25_30_only)

mask_tm_other_non_vae = (~mask_tm_close_to_vae) & (~mask_tm_vae_25_30_only)
mask_tp_other_non_vae = (~mask_tp_close_to_vae) & (~mask_tp_vae_25_30_only)

dist_excluded = np.concatenate([
    min_dist_tm_to_vae[mask_tm_excluded_non_vae],
    min_dist_tp_to_vae[mask_tp_excluded_non_vae],
])
dist_other = np.concatenate([
    min_dist_tm_to_vae[mask_tm_other_non_vae],
    min_dist_tp_to_vae[mask_tp_other_non_vae],
])

max_d = int(np.max(np.concatenate([dist_excluded, dist_other, np.array([10.0])])))
bins = np.arange(-0.5, max_d + 1.5, 1)

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

# Pannello 1: istogrammi sovrapposti
axes[0].hist(dist_other, bins=bins, alpha=0.55, color="steelblue", label="Altre (non escluse)", density=True)
axes[0].hist(dist_excluded, bins=bins, alpha=0.55, color="tomato", label="Escluse (close <= 10)", density=True)
axes[0].axvline(10, color="black", linestyle="--", linewidth=2, label="soglia close=10")
axes[0].set_title("Distribuzione distanza minima da VAE[25,30]")
axes[0].set_xlabel("Distanza minima da VAE[25,30]")
axes[0].set_ylabel("Densita`")
axes[0].legend()
axes[0].grid(alpha=0.2)

# Pannello 2: ECDF per confronto netto
def ecdf(x):
    x_sorted = np.sort(x)
    y = np.arange(1, x_sorted.size + 1) / x_sorted.size
    return x_sorted, y

x_other, y_other = ecdf(dist_other)
x_exc, y_exc = ecdf(dist_excluded)
axes[1].plot(x_other, y_other, color="steelblue", linewidth=2, label="Altre (non escluse)")
axes[1].plot(x_exc, y_exc, color="tomato", linewidth=2, label="Escluse (close <= 10)")
axes[1].axvline(10, color="black", linestyle="--", linewidth=2, label="soglia close=10")
axes[1].set_title("ECDF distanza minima da VAE[25,30]")
axes[1].set_xlabel("Distanza minima da VAE[25,30]")
axes[1].set_ylabel("Frazione cumulata")
axes[1].set_ylim(0, 1.02)
axes[1].legend()
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

print("Conteggi usati nel plot di conferma")
print(f"Escluse non-VAE: {dist_excluded.shape[0]}")
print(f"Altre non-VAE:   {dist_other.shape[0]}")
print(f"VAE [25,30] escluse dal confronto: {int(mask_tm_vae_25_30_only.sum() + mask_tp_vae_25_30_only.sum())}")